# AlphaTrain Pillar 2V: Pure Policy Iteration 2

**Self-play loop working.** V7 data from 2U ep8 (policy=1,763, MCTS@400=5,436).
1,250 games, 3.87M states, mean=6,594, dynamic sims (800-1400).

- Pure policy distillation (val_weight=0, rank_weight=0)
- Warm start from **2U epoch 8** (best policy ever)
- 10 epochs, eval bracket at 3/6/9

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `selfplay_v7_density_g098_ms200.pt.gz` — V7 tensor (278MB)
3. `pillar2u_epoch_8.pt` — best policy model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code (always fresh)
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy best policy model (2U ep8)
os.makedirs('/content/alphatrain/data', exist_ok=True)
print('Copying best policy model (2U ep8)...')
shutil.copy(f'{DRIVE}/pillar2u_epoch_8.pt', '/content/alphatrain/data/model.pt')
assert os.path.exists('/content/alphatrain/data/model.pt'), 'Model copy failed!'
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

# Decompress V7 data
print('Decompressing V7 data...')
t0 = time.time()
!gzip -dc {DRIVE}/selfplay_v7_density_g098_ms200.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'Data: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

# VERIFY
for f in ['model.pt', 'selfplay.pt']:
    path = f'/content/alphatrain/data/{f}'
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'  OK: {f} ({os.path.getsize(path)/1e6:.0f} MB)')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Pillar 2V: Pure Policy Iteration 2 — V7 data from stronger teacher
# Warm start from 2U epoch 8
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/selfplay.pt \
    --gpu-data --amp --compile \
    --value-bins 128 --value-channels 32 --value-hidden 512 --value-dropout 0.3 \
    --val-weight 0.0 --rank-weight 0.0 \
    --resume alphatrain/data/model.pt --warm-start \
    --epochs 10 --batch-size 24576 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2v_best.pt \
    --save-dir /content/checkpoints/pillar2v

In [ ]:
# Copy ALL epoch checkpoints to Drive for bracket eval
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/pillar2v/epoch_*.pt')):
    dst = f'{DRIVE}/pillar2v_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2v/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/pillar2v_{f}')